# Neo4j Query Graph Pipeline

In [29]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))

Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [30]:
# 1) Config and imports
import os
import re
import json
from pathlib import Path
import requests
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')
NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')

GPT_API_KEY = os.getenv('GPT_API_KEY')
GPT_MODEL = os.getenv('GPT_MODEL', 'gpt-5.4-mini')

BASE_DIR = Path.cwd()
ANNOT_DIR = BASE_DIR / 'annotation'


In [31]:
# 2) Neo4j helpers and candidate lists
def run_cypher(query, params=None):
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as _driver:
        with _driver.session(database=NEO4J_DB) as _session:
            return [r.data() for r in _session.run(query, params or {})]

def has_label(label):
    labels = [r['label'] for r in run_cypher('CALL db.labels() YIELD label RETURN label') if r.get('label')]
    return label in set(labels)

def has_property(prop):
    props = [r['propertyKey'] for r in run_cypher('CALL db.propertyKeys() YIELD propertyKey RETURN propertyKey') if r.get('propertyKey')]
    return prop in set(props)

def fetch_candidates():
    cities = [r['key'] for r in run_cypher('MATCH (c:City) RETURN c.key AS key') if r.get('key')]
    airports = [r['key'] for r in run_cypher('MATCH (a:Airport) RETURN a.key AS key') if r.get('key')]
    countries = [r['key'] for r in run_cypher('MATCH (c:Country) RETURN c.key AS key') if r.get('key')]
    doc_types = [r['v'] for r in run_cypher('MATCH (di:DocumentInstance) WHERE di.documentType IS NOT NULL RETURN DISTINCT di.documentType AS v') if r.get('v')]
    topic_keys = [r['v'] for r in run_cypher('MATCH (q:Questioning) WHERE q.topicKey IS NOT NULL RETURN DISTINCT q.topicKey AS v') if r.get('v')]
    biometric = []
    if has_label('BiometricCheck') and has_property('biometricModality'):
        biometric = [r['v'] for r in run_cypher('MATCH (b:BiometricCheck) WHERE b.biometricModality IS NOT NULL RETURN DISTINCT b.biometricModality AS v') if r.get('v')]
    return {
        'cities': sorted(cities),
        'airports': sorted(airports),
        'countries': sorted(countries),
        'doc_types': sorted(doc_types),
        'topic_keys': sorted(topic_keys),
        'biometric_modalities': sorted(biometric),
    }

candidates = fetch_candidates()


In [32]:
# 3) Load question text
def get_question_text(qnum: int) -> str:
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            q = (data.get('question') or '').strip()
            if q:
                return q
    raise FileNotFoundError(f'Question text not found for q_{qnum}')

QNUM = 26
QUESTION = get_question_text(QNUM)
QUESTION


'In which city in France are the most documents requested?'

In [33]:
# 4) LLM router: intents, mode, anchors
def llm_route(question: str, candidates: dict) -> dict:
    if not GPT_API_KEY:
        return {
            'intents': ['documentCheck'],
            'mode': 'single',
            'conditional': None,
            'scope': {'cities': [], 'airports': [], 'countries': []},
            'doc_types': [],
            'topic_keys': [],
            'biometric_modalities': [],
            'aggregate': None,
        }

    system = (
        'You route a question to intents and retrieval mode. '
        'Return ONLY JSON with keys: intents, mode, conditional, scope, doc_types, topic_keys, biometric_modalities, aggregate. '
        'Select anchors ONLY from provided candidates. '
        'Only include city/airport if explicitly named in the question (not just the word city). '
        'Choose intents from: documentCheck, questioning, biometric. '
        'mode must be one of: single, union, intersect, conditional, aggregate. '
        'If mode=conditional, set conditional to doc_given_topic or topic_given_doc. '
        'If the question is about most/least/how many/top/fewest, set mode=aggregate and fill aggregate. '
        'aggregate.group_by must be country/city/airport, metric one of documents/questions/biometric, order asc/desc, top_k integer. '
        'If not aggregate, set aggregate to null.'
    )

    user = {
        'question': question,
        'candidates': candidates
    }

    schema = {
        'type': 'object',
        'additionalProperties': False,
        'required': ['intents','mode','conditional','scope','doc_types','topic_keys','biometric_modalities','aggregate'],
        'properties': {
            'intents': {'type': 'array', 'items': {'type': 'string'}},
            'mode': {'type': 'string'},
            'conditional': {'type': 'string'},
            'scope': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['cities','airports','countries'],
                'properties': {
                    'cities': {'type': 'array', 'items': {'type': 'string'}},
                    'airports': {'type': 'array', 'items': {'type': 'string'}},
                    'countries': {'type': 'array', 'items': {'type': 'string'}}
                }
            },
            'doc_types': {'type': 'array', 'items': {'type': 'string'}},
            'topic_keys': {'type': 'array', 'items': {'type': 'string'}},
            'biometric_modalities': {'type': 'array', 'items': {'type': 'string'}},
            'aggregate': {
                'anyOf': [
                    {'type': 'null'},
                    {
                        'type': 'object',
                        'additionalProperties': False,
                        'required': ['group_by','metric','order','top_k'],
                        'properties': {
                            'group_by': {'type': 'string'},
                            'metric': {'type': 'string'},
                            'order': {'type': 'string'},
                            'top_k': {'type': 'integer'}
                        }
                    }
                ]
            }
        }
    }

    payload = {
        'model': GPT_MODEL,
        'input': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': json.dumps(user, ensure_ascii=False)}
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'route',
                'schema': schema,
                'strict': True
            }
        }
    }

    resp = requests.post(
        'https://api.openai.com/v1/responses',
        headers={'Authorization': f'Bearer {GPT_API_KEY}'},
        json=payload,
        timeout=60
    )
    if not resp.ok:
        raise RuntimeError(f'OpenAI API error {resp.status_code}: {resp.text[:500]}')
    data = resp.json()
    text = data.get('output_text') or ''
    if not text:
        out = []
        for item in data.get('output', []):
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    out.append(content.get('text', ''))
        text = ''.join(out).strip()
    if not text:
        raise ValueError('Empty LLM response')
    return json.loads(text)

route = llm_route(QUESTION, candidates)
route


{'intents': ['documentCheck'],
 'mode': 'aggregate',
 'conditional': '',
 'scope': {'cities': ['city_lyon',
   'city_marseille',
   'city_nice',
   'city_paris'],
  'airports': ['airport_cdg', 'airport_nce', 'airport_ory'],
  'countries': ['country_france']},
 'doc_types': ['cash',
  'foreign_passport',
  'hotel_booking',
  'payment_card',
  'return_ticket',
  'travel_insurance'],
 'topic_keys': ['accommodation_details',
  'cash_amount',
  'employment',
  'hotel_payment_card',
  'hotel_payment_status',
  'visit_duration',
  'visit_purpose'],
 'biometric_modalities': [],
 'aggregate': {'group_by': 'city',
  'metric': 'documents',
  'order': 'desc',
  'top_k': 1}}

In [34]:
# 5) Sanitize route output
def sanitize_route(route, candidates):
    allowed_intents = {'documentCheck','questioning','biometric'}
    allowed_modes = {'single','union','intersect','conditional','aggregate'}
    allowed_conditional = {'doc_given_topic','topic_given_doc'}
    allowed_metric = {'documents','questions','biometric'}
    allowed_group = {'country','city','airport'}
    allowed_order = {'asc','desc'}

    out = {
        'intents': [],
        'mode': route.get('mode','single'),
        'conditional': route.get('conditional'),
        'scope': {'cities': [], 'airports': [], 'countries': []},
        'doc_types': [],
        'topic_keys': [],
        'biometric_modalities': [],
        'aggregate': route.get('aggregate')
    }

    intents = route.get('intents') if isinstance(route, dict) else []
    out['intents'] = [i for i in intents if i in allowed_intents] or ['documentCheck']

    scope = route.get('scope', {}) if isinstance(route, dict) else {}
    for k in ['cities','airports','countries']:
        vals = scope.get(k, []) if isinstance(scope, dict) else []
        allowed = set(candidates.get(k, []))
        out['scope'][k] = sorted({v for v in vals if v in allowed})

    for k in ['doc_types','topic_keys','biometric_modalities']:
        vals = route.get(k, []) if isinstance(route, dict) else []
        allowed = set(candidates.get(k, []))
        out[k] = sorted({v for v in vals if v in allowed})

    if out['mode'] not in allowed_modes:
        out['mode'] = 'single'

    if out['mode'] == 'union' and len(out['intents']) < 2:
        out['mode'] = 'single'

    if out['mode'] in {'conditional','intersect'}:
        if not ({'documentCheck','questioning'} <= set(out['intents'])):
            out['mode'] = 'single'

    if out['mode'] == 'conditional':
        if out['conditional'] not in allowed_conditional:
            out['conditional'] = 'doc_given_topic'

    agg = out.get('aggregate')
    if out['mode'] != 'aggregate':
        out['aggregate'] = None
    else:
        if not isinstance(agg, dict):
            out['mode'] = 'single'
            out['aggregate'] = None
        else:
            group_by = agg.get('group_by')
            metric = agg.get('metric')
            order = agg.get('order','desc')
            top_k = int(agg.get('top_k', 1))
            if group_by not in allowed_group or metric not in allowed_metric:
                out['mode'] = 'single'
                out['aggregate'] = None
            else:
                out['aggregate'] = {
                    'group_by': group_by,
                    'metric': metric,
                    'order': order if order in allowed_order else 'desc',
                    'top_k': max(1, top_k)
                }

    return out

ROUTE = sanitize_route(route, candidates)
ROUTE


{'intents': ['documentCheck'],
 'mode': 'aggregate',
 'conditional': '',
 'scope': {'cities': ['city_lyon',
   'city_marseille',
   'city_nice',
   'city_paris'],
  'airports': ['airport_cdg', 'airport_nce', 'airport_ory'],
  'countries': ['country_france']},
 'doc_types': ['cash',
  'foreign_passport',
  'hotel_booking',
  'payment_card',
  'return_ticket',
  'travel_insurance'],
 'topic_keys': ['accommodation_details',
  'cash_amount',
  'employment',
  'hotel_payment_card',
  'hotel_payment_status',
  'visit_duration',
  'visit_purpose'],
 'biometric_modalities': [],
 'aggregate': {'group_by': 'city',
  'metric': 'documents',
  'order': 'desc',
  'top_k': 1}}

In [35]:
# 6) Build query graph object
QUERY_GRAPH = {
    'question': QUESTION,
    **ROUTE
}
QUERY_GRAPH


{'question': 'In which city in France are the most documents requested?',
 'intents': ['documentCheck'],
 'mode': 'aggregate',
 'conditional': '',
 'scope': {'cities': ['city_lyon',
   'city_marseille',
   'city_nice',
   'city_paris'],
  'airports': ['airport_cdg', 'airport_nce', 'airport_ory'],
  'countries': ['country_france']},
 'doc_types': ['cash',
  'foreign_passport',
  'hotel_booking',
  'payment_card',
  'return_ticket',
  'travel_insurance'],
 'topic_keys': ['accommodation_details',
  'cash_amount',
  'employment',
  'hotel_payment_card',
  'hotel_payment_status',
  'visit_duration',
  'visit_purpose'],
 'biometric_modalities': [],
 'aggregate': {'group_by': 'city',
  'metric': 'documents',
  'order': 'desc',
  'top_k': 1}}

In [36]:
# 7) Compile query graph to Cypher
SCOPE_MATCH = '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)
    WHEN has_countries THEN match_country
    ELSE true
END
'''

def compile_aggregate_cypher(qg):
    agg = qg.get('aggregate') or {}
    group_by = agg.get('group_by')
    metric = agg.get('metric')
    order = agg.get('order','desc')
    top_k = int(agg.get('top_k', 1))

    if group_by == 'country':
        group_expr = 'coalesce(co, co2, co3)'
        group_key = 'country.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS country"
        group_where = 'WHERE country IS NOT NULL'
        group_name = 'country'
    elif group_by == 'city':
        group_expr = 'coalesce(c, c2)'
        group_key = 'city.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS city"
        group_where = 'WHERE city IS NOT NULL'
        group_name = 'city'
    elif group_by == 'airport':
        group_expr = 'a'
        group_key = 'airport.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS airport"
        group_where = 'WHERE airport IS NOT NULL'
        group_name = 'airport'
    else:
        raise ValueError('Unsupported aggregate group_by')

    if metric == 'documents':
        metric_match = 'MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)'
        metric_where = 'WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)'
        metric_expr = 'count(*) AS metric'
    elif metric == 'questions':
        metric_match = 'MATCH (e)-[:hasQuestioning]->(q:Questioning)'
        metric_where = 'WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)'
        metric_expr = 'count(*) AS metric'
    elif metric == 'biometric':
        metric_match = 'MATCH (e)-[:hasBiometricCheck]->(b:BiometricCheck)'
        metric_where = 'WHERE (size($biometric_modalities)=0 OR b.biometricModality IN $biometric_modalities)'
        metric_expr = 'count(*) AS metric'
    else:
        raise ValueError('Unsupported aggregate metric')

    limit_clause = f"LIMIT {top_k}" if top_k > 0 else ''

    return f'''
{SCOPE_MATCH}
{group_with}
{group_where}
{metric_match}
{metric_where}
RETURN {group_key} AS {group_name}, {metric_expr}
ORDER BY metric {order}
{limit_clause}
'''.strip()


def compile_documentcheck_cypher(qg):
    return f'''
{SCOPE_MATCH}
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type
WHERE (size($doc_types)=0 OR di_type IN $doc_types)
CALL (e,dc,di,di_type,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_questioning_cypher(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
WITH e,a,c,co,c2,co2,co3,q,q.topicKey AS q_topic
CALL (e,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_biometric_cypher(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasBiometricCheck]->(b:BiometricCheck)
WHERE (size($biometric_modalities)=0 OR b.biometricModality IN $biometric_modalities)
WITH e,a,c,co,c2,co2,co3,b,b.biometricModality AS b_mod
CALL (e,b,b_mod,a,c,co,c2,co2,co3) {{
  WITH e,b WHERE e IS NOT NULL AND b IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasBiometricCheck' AS p, labels(b)[0] AS o_label, b.key AS o_key
  UNION
  WITH b,b_mod WHERE b IS NOT NULL AND b_mod IS NOT NULL
  RETURN DISTINCT labels(b)[0] AS s_label, b.key AS s_key, 'biometricModality' AS p, 'Literal' AS o_label, b_mod AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_doc_given_topic(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_topic_given_doc(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_intersect_doc_questioning(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_to_cypher(qg):
    mode = qg.get('mode')
    intents = qg.get('intents') or ['documentCheck']
    if mode == 'aggregate':
        return compile_aggregate_cypher(qg)
    if mode == 'conditional':
        if qg.get('conditional') == 'topic_given_doc':
            return compile_topic_given_doc(qg)
        return compile_doc_given_topic(qg)
    if mode == 'intersect' and {'documentCheck','questioning'} <= set(intents):
        return compile_intersect_doc_questioning(qg)
    if len(intents) == 1:
        if intents[0] == 'documentCheck':
            return compile_documentcheck_cypher(qg)
        if intents[0] == 'questioning':
            return compile_questioning_cypher(qg)
        if intents[0] == 'biometric':
            return compile_biometric_cypher(qg)
    return None

cypher = compile_to_cypher(QUERY_GRAPH)
cypher


'MATCH (e:Encounter)\nOPTIONAL MATCH (e)-[:atAirport]->(a:Airport)\nOPTIONAL MATCH (e)-[:atCity]->(c:City)\nOPTIONAL MATCH (e)-[:atCountry]->(co:Country)\nOPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)\nOPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)\nOPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)\nWITH e,a,c,co,c2,co2,co3,\n     (size($airports) > 0) AS has_airports,\n     (size($cities) > 0) AS has_cities,\n     (size($countries) > 0) AS has_countries,\n     (a IS NOT NULL AND a.key IN $airports) AS match_airport,\n     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,\n     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country\nWHERE CASE\n    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)\n    WHEN has_countries THEN match_country\n    ELSE true\nEND\n\nWITH e,a,c,co,c2,co2,co3, coalesce(c, c2) AS city\

In [37]:
# 8) Execute retrieval
params = {
    'cities': QUERY_GRAPH['scope']['cities'],
    'airports': QUERY_GRAPH['scope']['airports'],
    'countries': QUERY_GRAPH['scope']['countries'],
    'doc_types': QUERY_GRAPH['doc_types'],
    'topic_keys': QUERY_GRAPH['topic_keys'],
    'biometric_modalities': QUERY_GRAPH['biometric_modalities'],
}

rows = []
if QUERY_GRAPH['mode'] == 'union' and len(QUERY_GRAPH['intents']) > 1:
    seen = set()
    for intent in QUERY_GRAPH['intents']:
        qg_one = dict(QUERY_GRAPH)
        qg_one['intents'] = [intent]
        qg_one['mode'] = 'single'
        c = compile_to_cypher(qg_one)
        for r in run_cypher(c, params):
            t = (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
            if t not in seen:
                seen.add(t)
                rows.append(r)
else:
    rows = run_cypher(cypher, params) if cypher else []

rows[:5]


[{'city': 'city_paris', 'metric': 13}]

In [38]:
# 9) Postprocess / evaluate (optional)
def triples_to_set(rows):
    return {
        (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
        for r in rows
        if all(k in r for k in ['s_label','s_key','p','o_label','o_key'])
    }

def load_gold(qnum: int):
    triples = []
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            for t in data.get('triples', []):
                s = t.get('s', {})
                o = t.get('o', {})
                triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
    return set(triples)

if rows and all(k in rows[0] for k in ['s_label','s_key','p','o_label','o_key']):
    retrieved = triples_to_set(rows)
    gold = load_gold(QNUM)
    tp = len(retrieved & gold)
    precision = tp / len(retrieved) if retrieved else 0.0
    recall = tp / len(gold) if gold else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    print('retrieved', len(retrieved))
    print('gold', len(gold))
    print('precision', precision)
    print('recall', recall)
    print('f1', f1)
else:
    print('Aggregate output:', rows[:5])


Aggregate output: [{'city': 'city_paris', 'metric': 13}]
